# Backtesting Your Strategy with SportsQuant

**Prove your strategy works before risking money**

A strategy that looks great in hindsight often fails in practice. Backtesting lets you simulate how a strategy would have performed on historical data. Walk-forward validation goes further — it prevents look-ahead bias and overfitting by using only past data for each prediction.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.engine import decide_over_under
from sportsquant.core.betting.kelly import KellyCalculator
from sportsquant.core.betting.metrics import (
    calculate_performance_metrics,
    calculate_sharpe_ratio,
    calculate_max_drawdown,
    calculate_profit_factor,
)

BACKTEST_AVAILABLE = False
try:
    from sportsquant.core.backtest.regime import (
        WalkForwardBacktest,
        WalkForwardConfig,
        RegimePeriod,
        SensitivityAnalyzer,
    )
    BACKTEST_AVAILABLE = True
    print("Imports successful. Let's backtest!")
except ImportError as e:
    print(f"⚠️ Some backtest modules not available: {e}")
    print("Running in demo mode with mock data only.")
    print("Core betting imports loaded successfully.")

Imports successful. Let's backtest!


## 1. Generate Synthetic Historical Data

We create synthetic NBA prop data to demonstrate the backtesting framework.

## 2. Create a Simple Value Betting Strategy

We'll define a strategy: **Bet OVER when our model's probability exceeds implied probability by at least 2% (edge threshold).**

In [2]:
np.random.seed(42)

# Generate synthetic historical data: 200 NBA games with props
n_games = 200

historical_data = pd.DataFrame({
    "game_date": pd.date_range("2024-10-22", periods=n_games, freq="D"),
    "player_id": np.random.choice(range(1, 31), n_games),
    "player_name": np.random.choice(
        ["LeBron", "Curry", "Doncic", "Jokic", "Giannis", "Tatum", "Embiid", "SGA",
         "Durant", "Davis", "Young", "Lillard", "Morant", "Booker", "Mitchell",
         "Harden", "Butler", "Sabonis", "Fox", "Edwards", "Brunson", "Haliburton",
         "Markkanen", "Ingram", "LaVine", "Murray", "Trae", "Cade", "Wemby", "Bane"],
        n_games
    ),
    "stat": np.random.choice(["Points", "Rebounds", "Assists"], n_games, p=[0.5, 0.25, 0.25]),
    "line": np.round(np.random.normal(20, 8, n_games) * 2) / 2,
    "odds_american_over": np.random.choice([-110, -115, -105, -108], n_games),
    "odds_american_under": np.random.choice([-110, -105, -115, -112], n_games),
})

# Our model's estimated probability (slightly better than random due to "skill")
historical_data["model_prob_over"] = np.clip(
    np.random.beta(52, 48, n_games), 0.35, 0.70
)

# Actual outcome: correlated with model but with noise
noise = np.random.normal(0, 0.05, n_games)
historical_data["actual_prob_over"] = np.clip(historical_data["model_prob_over"] + noise, 0.01, 0.99)
historical_data["actual_over"] = (np.random.random(n_games) < historical_data["actual_prob_over"]).astype(int)

# Compute edge and implied probability
historical_data["implied_prob"] = historical_data["odds_american_over"].apply(
    lambda x: Odds(american=x).implied_prob()
)
historical_data["edge"] = historical_data["model_prob_over"] - historical_data["implied_prob"]

print(f"Generated {len(historical_data)} historical prop lines")
print(f"Average model edge: {historical_data['edge'].mean():.4f}")
print(f"Positive edge rate: {(historical_data['edge'] > 0).mean():.1%}")
historical_data.head()

Generated 200 historical prop lines
Average model edge: -0.0029
Positive edge rate: 46.0%


,game_date,player_id,player_name,stat,line,odds_american_over,odds_american_under,model_prob_over,actual_prob_over,actual_over,implied_prob,edge
0,2024-10-22,7,Embiid,Points,35.5,-105,-110,0.528276,0.542403,0,0.512195,0.016081
1,2024-10-23,20,Morant,Rebounds,28.5,-115,-112,0.516189,0.480796,0,0.534884,-0.018695
2,2024-10-24,29,Mitchell,Points,5.5,-110,-115,0.519156,0.480070,0,0.523810,-0.004654
3,2024-10-25,15,Young,Points,16.5,-115,-105,0.531622,0.575602,1,0.534884,-0.003261
4,2024-10-26,11,Wemby,Rebounds,29.5,-105,-105,0.399751,0.430108,1,0.512195,-0.112444


## 3. Basic Backtest: ROI, Win Rate, Sharpe

Run a simple backtest: bet on every prop where edge > 2%.

In [3]:
edge_threshold = 0.02

def run_basic_backtest(data, threshold):
    """Run a simple backtest with an edge threshold."""
    bets = data[data["edge"] > threshold].copy()
    
    pnls = []
    evs = []
    outcomes = []
    stakes = []
    
    for _, row in bets.iterrows():
        odds_over = Odds(american=int(row["odds_american_over"]))
        odds_under = Odds(american=int(row["odds_american_under"]))
        
        try:
            decision = decide_over_under(
                line=row["line"],
                p_over=row["model_prob_over"],
                odds_over=odds_over,
                odds_under=odds_under,
                true_prob_over=row["model_prob_over"],
                true_prob_under=1.0 - row["model_prob_over"],
            )
            stake = 1.0
            win = (decision.side == "over" and row["actual_over"] == 1) or \
                  (decision.side == "under" and row["actual_over"] == 0)
            pnl = (decision.decimal_odds - 1) * stake if win else -stake
        except Exception:
            # Fallback: simple calculation
            implied = odds_over.implied_prob()
            our_prob = row["model_prob_over"]
            stake = 1.0
            win = row["actual_over"] == 1 if our_prob > implied else row["actual_over"] == 0
            decimal_odds = odds_over.to_decimal()
            pnl = (decimal_odds - 1) * stake if win else -stake
        
        pnls.append(pnl)
        evs.append(row["model_prob_over"] * (odds_over.to_decimal() - 1) - (1 - row["model_prob_over"]))
        outcomes.append(win)
        stakes.append(stake)
    
    if not pnls:
        return None
    
    try:
        metrics = calculate_performance_metrics(
            pnl_series=pd.Series(pnls),
            ev_series=pd.Series(evs),
            outcome_series=pd.Series(outcomes),
            stake_series=pd.Series(stakes),
        )
        return metrics, pnls, outcomes
    except Exception as e:
        # Fallback: return a dict-like result
        total_pnl = sum(pnls)
        win_rate = sum(outcomes) / len(outcomes) if outcomes else 0
        sharpe = np.mean(pnls) / np.std(pnls) * np.sqrt(252) if np.std(pnls) > 0 else 0
        class FallbackMetrics:
            pass
        m = FallbackMetrics()
        m.core = FallbackMetrics()
        m.core.total_bets = len(pnls)
        m.core.win_rate = win_rate
        m.core.total_pnl = total_pnl
        m.risk_adjusted = FallbackMetrics()
        m.risk_adjusted.sharpe_ratio = sharpe
        m.risk_adjusted.sortino_ratio = 0
        m.risk_adjusted.profit_factor = sum(p for p in pnls if p > 0) / abs(sum(p for p in pnls if p < 0)) if sum(p for p in pnls if p < 0) != 0 else 0
        m.drawdown = FallbackMetrics()
        m.drawdown.max_drawdown = max(0, -min(np.cumsum(pnls) - np.maximum.accumulate(np.cumsum(pnls))))
        m.streaks = FallbackMetrics()
        m.streaks.max_win_streak = 0
        m.streaks.max_loss_streak = 0
        return m, pnls, outcomes

result = run_basic_backtest(historical_data, edge_threshold)
if result:
    metrics, pnls, outcomes = result
    # Handle both nested and flat metric access
    total_bets = getattr(getattr(metrics, 'core', metrics), 'total_bets', 0)
    win_rate = getattr(getattr(metrics, 'core', metrics), 'win_rate', 0)
    total_pnl = getattr(getattr(metrics, 'core', metrics), 'total_pnl', 0)
    sharpe = getattr(getattr(metrics, 'risk_adjusted', metrics), 'sharpe_ratio', 0)
    sortino = getattr(getattr(metrics, 'risk_adjusted', metrics), 'sortino_ratio', 0)
    max_dd = getattr(getattr(metrics, 'drawdown', metrics), 'max_drawdown', 0)
    pf = getattr(getattr(metrics, 'risk_adjusted', metrics), 'profit_factor', 0)
    max_ws = getattr(getattr(metrics, 'streaks', metrics), 'max_win_streak', 0)
    max_ls = getattr(getattr(metrics, 'streaks', metrics), 'max_loss_streak', 0)
    
    print(f"=== Basic Backtest (edge > {edge_threshold:.0%}) ===")
    print(f"Total bets: {total_bets}")
    print(f"Win rate: {win_rate:.1%}")
    print(f"Total P&L: ${total_pnl:.2f} per $1 unit")
    print(f"Sharpe ratio: {sharpe:.2f}")
    print(f"Sortino ratio: {sortino:.2f}")
    print(f"Max drawdown: {max_dd:.2f}")
    print(f"Profit factor: {pf:.2f}")
    print(f"Max win streak: {max_ws}")
    print(f"Max loss streak: {max_ls}")
else:
    print("No bets met the edge threshold.")
    result = None

=== Basic Backtest (edge > 2%) ===
Total bets: 63
Win rate: 54.0%
Total P&L: $2.14 per $1 unit
Sharpe ratio: 0.56
Sortino ratio: 0.00
Max drawdown: 5.70
Profit factor: 1.07
Max win streak: 5
Max loss streak: 4


In [4]:
# Visualize equity curve
if result:
    cumulative = np.cumsum(pnls)
    running_max = np.maximum.accumulate(cumulative)
    drawdown = running_max - cumulative
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)
    
    # Equity curve
    axes[0].plot(cumulative, color="steelblue", linewidth=2)
    axes[0].fill_between(range(len(cumulative)), 0, cumulative, where=cumulative > 0, alpha=0.2, color="green")
    axes[0].fill_between(range(len(cumulative)), 0, cumulative, where=cumulative < 0, alpha=0.2, color="red")
    axes[0].set_ylabel("Cumulative P&L ($)")
    axes[0].set_title("Equity Curve")
    
    # Drawdown
    axes[1].fill_between(range(len(drawdown)), 0, drawdown, color="red", alpha=0.3)
    axes[1].set_ylabel("Drawdown ($)")
    axes[1].set_title("Drawdown")
    
    # Rolling Sharpe (20-bet window)
    pnl_arr = np.array(pnls)
    window = 20
    rolling_sharpe = []
    for i in range(len(pnl_arr)):
        start = max(0, i - window)
        window_pnl = pnl_arr[start:i+1]
        if len(window_pnl) > 2 and np.std(window_pnl) > 0:
            rolling_sharpe.append(np.mean(window_pnl) / np.std(window_pnl) * np.sqrt(252))
        else:
            rolling_sharpe.append(0)
    
    axes[2].plot(rolling_sharpe, color="purple", linewidth=1.5)
    axes[2].axhline(y=0, color="black", linestyle="--", alpha=0.5)
    axes[2].set_ylabel("Rolling Sharpe (20-bet)")
    axes[2].set_xlabel("Bet Number")
    axes[2].set_title("Rolling Sharpe Ratio")
    
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No backtest results to visualize. Equity curve skipped.")

/tmp/ipykernel_1964232/1838014782.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Walk-Forward Validation

Walk-forward validation is the **gold standard**. Instead of fitting parameters on all data, it:
1. Trains on the first N games (e.g., 82)
2. Tests on the next M games (e.g., 20)
3. Expands the training window and repeats

This mimics real-world deployment: you can only use *past* data to predict *future* outcomes.

In [5]:
# Prepare features for walk-forward
if not BACKTEST_AVAILABLE:
    print("⚠️ Walk-forward backtest modules not available — showing mock walk-forward results")
    print("With real modules, this would run walk-forward validation using GradientBoostingClassifier.")
else:
    from sklearn.ensemble import GradientBoostingClassifier

    # Build feature matrix from historical data
    feature_cols = ["model_prob_over", "edge", "implied_prob", "line"]
    X = historical_data[feature_cols].fillna(0).copy()
    y = historical_data["actual_over"]

    # Run walk-forward backtest
    wfb = WalkForwardBacktest(
        features=X,
        y=y,
        train_window=82,
        test_window=20,
    )

    results = wfb.run(
        model_factory=lambda: GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
        feature_names=feature_cols,
    )

    print(f"Walk-Forward Results: {results.total_folds} folds")
    print(f"\nOverall metrics:")
    for k, v in results.overall_metrics.items():
        if isinstance(v, (int, float)):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")

Walk-Forward Results: 99 folds

Overall metrics:
  log_loss_mean: 0.7733
  log_loss_std: 0.1053
  log_loss_min: 0.5019
  log_loss_max: 0.9823
  accuracy_mean: 0.5167
  accuracy_std: 0.1101
  accuracy_min: 0.3000
  accuracy_max: 0.7500
  total_folds: 99.0000
  total_samples: 1980.0000


In [6]:
# Compare walk-forward vs naive backtest
if not BACKTEST_AVAILABLE or 'results' not in dir() or results is None:
    print("⚠️ Walk-forward results not available — showing mock fold comparison")
    # Generate mock fold data
    np.random.seed(42)
    n_folds = 5
    fold_accuracies = np.random.uniform(0.52, 0.62, n_folds)
    fold_log_losses = np.random.uniform(0.65, 0.72, n_folds)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].bar(range(n_folds), fold_accuracies, color="steelblue", alpha=0.7)
    axes[0].axhline(y=0.5, color="red", linestyle="--", label="Random (50%)")
    axes[0].set_xlabel("Fold")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Walk-Forward: Accuracy by Fold (Mock)")
    axes[0].legend()
    axes[1].bar(range(n_folds), fold_log_losses, color="coral", alpha=0.7)
    axes[1].axhline(y=-np.log(0.5), color="red", linestyle="--", label="Random baseline")
    axes[1].set_xlabel("Fold")
    axes[1].set_ylabel("Log Loss")
    axes[1].set_title("Walk-Forward: Log Loss by Fold (Mock)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    fold_accuracies = []
    fold_log_losses = []

    for fold in results.fold_results:
        if "error" not in fold.metrics:
            fold_accuracies.append(fold.metrics.get("accuracy", 0))
            fold_log_losses.append(fold.metrics.get("log_loss", 0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy by fold
    axes[0].bar(range(len(fold_accuracies)), fold_accuracies, color="steelblue", alpha=0.7)
    axes[0].axhline(y=0.5, color="red", linestyle="--", label="Random (50%)")
    axes[0].set_xlabel("Fold")
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Walk-Forward: Accuracy by Fold")
    axes[0].legend()

    # Log loss by fold
    axes[1].bar(range(len(fold_log_losses)), fold_log_losses, color="coral", alpha=0.7)
    axes[1].axhline(y=-np.log(0.5), color="red", linestyle="--", label="Random baseline")
    axes[1].set_xlabel("Fold")
    axes[1].set_ylabel("Log Loss")
    axes[1].set_title("Walk-Forward: Log Loss by Fold (lower is better)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

/tmp/ipykernel_1964232/1216869357.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Sensitivity Analysis: Finding Optimal Edge Threshold

The `SensitivityAnalyzer` sweeps parameter values to find the optimal configuration.

In [7]:
# Sensitivity analysis on edge threshold
thresholds = [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.08, 0.10]
sensitivity_results = []

for thresh in thresholds:
    bt_result = run_basic_backtest(historical_data, thresh)
    if bt_result:
        m, _, _ = bt_result
        # Handle both nested and flat metric access
        total_bets = getattr(getattr(m, 'core', m), 'total_bets', 0)
        win_rate = getattr(getattr(m, 'core', m), 'win_rate', 0)
        total_pnl = getattr(getattr(m, 'core', m), 'total_pnl', 0)
        sharpe = getattr(getattr(m, 'risk_adjusted', m), 'sharpe_ratio', 0)
        pf = getattr(getattr(m, 'risk_adjusted', m), 'profit_factor', 0)
        max_dd = getattr(getattr(m, 'drawdown', m), 'max_drawdown', 0)
        sensitivity_results.append({
            "threshold": thresh,
            "n_bets": total_bets,
            "win_rate": win_rate,
            "total_pnl": total_pnl,
            "sharpe": sharpe,
            "profit_factor": pf,
            "max_drawdown": max_dd,
        })

sens_df = pd.DataFrame(sensitivity_results)
sens_df.style.format({
    "threshold": "{:.2f}", "win_rate": "{:.1%}",
    "total_pnl": "{:+.2f}", "sharpe": "{:.2f}", "profit_factor": "{:.2f}", "max_drawdown": "{:.2f}"
})

,threshold,n_bets,win_rate,total_pnl,sharpe,profit_factor,max_drawdown
0,0.00,92,52.2%,-0.13,-0.02,1.00,8.47
1,0.01,79,51.9%,-0.40,-0.08,0.99,6.79
2,0.02,63,54.0%,+2.14,0.56,1.07,5.70
3,0.03,53,52.8%,+0.68,0.21,1.03,6.55
4,0.04,39,59.0%,+5.13,2.18,1.32,3.50
5,0.05,27,59.3%,+3.70,2.26,1.34,1.35
6,0.06,25,56.0%,+1.82,1.19,1.17,2.07
7,0.08,13,53.8%,+0.44,0.54,1.07,3.16
8,0.10,3,33.3%,-1.07,-5.11,0.46,1.00


In [8]:
try:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # P&L vs threshold
    axes[0].plot(sens_df["threshold"], sens_df["total_pnl"], "o-", color="steelblue", linewidth=2)
    axes[0].axhline(y=0, color="red", linestyle="--")
    axes[0].set_xlabel("Edge Threshold")
    axes[0].set_ylabel("Total P&L ($)")
    axes[0].set_title("P&L by Edge Threshold")

    # Win rate vs threshold
    axes[1].plot(sens_df["threshold"], sens_df["win_rate"], "o-", color="green", linewidth=2)
    axes[1].axhline(y=0.5, color="red", linestyle="--", label="Breakeven")
    axes[1].set_xlabel("Edge Threshold")
    axes[1].set_ylabel("Win Rate")
    axes[1].set_title("Win Rate by Edge Threshold")
    axes[1].legend()

    # Sharpe vs threshold
    axes[2].plot(sens_df["threshold"], sens_df["sharpe"], "o-", color="purple", linewidth=2)
    axes[2].axhline(y=0, color="red", linestyle="--")
    axes[2].set_xlabel("Edge Threshold")
    axes[2].set_ylabel("Sharpe Ratio")
    axes[2].set_title("Sharpe Ratio by Edge Threshold")

    plt.tight_layout()
    plt.show()
except NameError as e:
    print(f"⚠️ Sensitivity visualization skipped: {e}")
    print("Run the sensitivity analysis cell first.")

/tmp/ipykernel_1964232/1560167069.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Regime Detection

Markets go through different regimes (high-scoring, low-scoring, high-variance, etc.). Testing across regimes shows if your strategy is robust.

In [9]:
# Define synthetic regime periods
if not BACKTEST_AVAILABLE:
    print("⚠️ Walk-forward backtest modules not available — showing mock regime results")
    print("With real modules, this would run regime-aware walk-forward validation.")
else:
    regime_periods = [
        RegimePeriod(start_date="0", end_date="60", regime_type="normal", description="Normal scoring"),
        RegimePeriod(start_date="61", end_date="120", regime_type="high_variance", description="High variance period"),
        RegimePeriod(start_date="121", end_date="200", regime_type="efficient", description="Efficient market"),
    ]

    # Run walk-forward with regime awareness
    regime_results = wfb.run_with_regimes(
        model_factory=lambda: GradientBoostingClassifier(n_estimators=50, max_depth=3, random_state=42),
        regimes=regime_periods,
    )

    print(f"Regime-aware results: {regime_results.total_folds} folds")
    print(f"\nRegime-specific metrics:")
    for regime, metrics in regime_results.regime_metrics.items():
        print(f"\n  {regime}:")
        for k, v in metrics.items():
            if isinstance(v, (int, float)):
                print(f"    {k}: {v:.4f}")
            else:
                print(f"    {k}: {v}")

    print(f"\nCross-regime comparison:")
    for comp_key, comp_val in regime_results.cross_regime_comparison.items():
        print(f"  {comp_key}: {comp_val}")

Regime-aware results: 99 folds

Regime-specific metrics:

  unknown:
    log_loss_mean: 0.6320
    log_loss_std: 0.0446
    log_loss_min: 0.5728
    log_loss_max: 0.7065
    accuracy_mean: 0.6562
    accuracy_std: 0.0464
    accuracy_min: 0.6000
    accuracy_max: 0.7500
    total_folds: 8.0000
    total_samples: 160.0000

  normal:
    log_loss_mean: 0.7857
    log_loss_std: 0.0999
    log_loss_min: 0.5019
    log_loss_max: 0.9823
    accuracy_mean: 0.5044
    accuracy_std: 0.1055
    accuracy_min: 0.3000
    accuracy_max: 0.7500
    total_folds: 91.0000
    total_samples: 1820.0000

Cross-regime comparison:
  unknown_vs_normal: {'diff': -0.15372285078807724, 'regime_a_advantage': True}


In [10]:
try:
    regime_names = list(regime_results.regime_metrics.keys())
    regime_ll = []
    regime_acc = []
    for name in regime_names:
        m = regime_results.regime_metrics[name]
        regime_ll.append(m.get("log_loss_mean", 0))
        regime_acc.append(m.get("accuracy_mean", 0))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    colors = ["steelblue", "coral", "forestgreen"]
    axes[0].bar(regime_names, regime_ll, color=colors[:len(regime_names)], alpha=0.7)
    axes[0].set_ylabel("Mean Log Loss")
    axes[0].set_title("Log Loss by Market Regime (lower is better)")

    axes[1].bar(regime_names, regime_acc, color=colors[:len(regime_names)], alpha=0.7)
    axes[1].axhline(y=0.5, color="red", linestyle="--", label="Random")
    axes[1].set_ylabel("Mean Accuracy")
    axes[1].set_title("Accuracy by Market Regime")
    axes[1].legend()

    plt.tight_layout()
    plt.show()
except NameError:
    print("⚠️ Regime visualization skipped — regime_results not available")
    print("With real modules, this would show regime-specific performance metrics.")

/tmp/ipykernel_1964232/1447974839.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Key Takeaway

> **Walk-forward validation is the gold standard; never trust a simple backtest.**

1. A simple backtest can look great due to overfitting or look-ahead bias
2. Walk-forward trains only on *past* data, mimicking real deployment
3. Sensitivity analysis finds optimal parameters without manual tuning
4. Regime detection shows whether your strategy works in *all* market conditions
5. Always check: ROI, Sharpe ratio, max drawdown, and profit factor together
6. A strategy that only works in one regime is fragile — diversify your approach